<a href="https://colab.research.google.com/github/kaplunaleks/CSsync/blob/master/CII_BlagovisnayaVO_lab1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **КОД ДЛЯ ПРОГНОЗИРОВАНИЯ УСПЕШНОСТИ ИГР НА РЫНКЕ**

---



1. УСТАНОВКА И ИМПОРТ БИБЛИОТЕК

In [13]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
import warnings
warnings.filterwarnings('ignore')

print("="*80)
print("ПРОГНОЗИРОВАНИЕ ПРОДАЖ ВИДЕОИГР В ЯПОНИИ")
print("="*80)

ПРОГНОЗИРОВАНИЕ ПРОДАЖ ВИДЕОИГР В ЯПОНИИ


2. ЗАГРУЗКА ДАННЫХ

In [14]:
print("\n[1/10] Загрузка данных...")

# Путь к файлу в Google Colab
df = pd.read_csv('/content/sample_data/Video_Games.csv')

print(f"  ✓ Загружено записей: {df.shape[0]}")
print(f"  ✓ Количество признаков: {df.shape[1]}")
print(f"\nСтруктура данных:")
print(df.head())
print(f"\nИнформация о пропущенных значениях:")
print(df.isnull().sum())


[1/10] Загрузка данных...
  ✓ Загружено записей: 11703
  ✓ Количество признаков: 15

Структура данных:
                      Name Platform  Year_of_Release       Genre  \
0          Rapala Trophies      PSP           2006.0      Sports   
1  New Super Mario Bros. U     WiiU           2012.0    Platform   
2                   Robots      PS2           2005.0      Action   
3           Hamster Club 3      GBA           2002.0  Simulation   
4             Formula 1 06      PS2           2006.0      Racing   

                     Publisher  NA_Sales  EU_Sales  JP_Sales  Other_Sales  \
0                   Activision      0.04      0.00      0.00         0.00   
1                     Nintendo      2.30      1.34      1.27         0.32   
2                Vivendi Games      0.18      0.14      0.00         0.05   
3                      Jorudan      0.00      0.00      0.35         0.01   
4  Sony Computer Entertainment      0.00      0.00      0.04         0.00   

   Critic_Score  Critic_

3. ПРЕДОБРАБОТКА ДАННЫХ

In [15]:
print("\n[2/10] Предобработка данных...")

# Создание копии для работы
data = df.copy()

# Удаление записей без целевой переменной (JP_Sales)
initial_count = len(data)
data = data.dropna(subset=['JP_Sales'])
print(f"  ✓ Удалено записей без JP_Sales: {initial_count - len(data)}")

# Удаление записей с критическими пропусками
data = data.dropna(subset=['Name', 'Platform', 'Genre'])
print(f"  ✓ Записей после удаления критических пропусков: {len(data)}")

# Заполнение пропущенных значений
# Year_of_Release - медианой
data['Year_of_Release'].fillna(data['Year_of_Release'].median(), inplace=True)

# Publisher - специальной категорией
data['Publisher'].fillna('Unknown', inplace=True)

# Developer - специальной категорией
data['Developer'].fillna('Unknown', inplace=True)

# Rating - специальной категорией
data['Rating'].fillna('Unknown', inplace=True)

# Critic_Score и User_Score - медианами
data['Critic_Score'].fillna(data['Critic_Score'].median(), inplace=True)
data['Critic_Count'].fillna(0, inplace=True)
data['User_Score'].fillna(data['User_Score'].median(), inplace=True)
data['User_Count'].fillna(0, inplace=True)

# Продажи в других регионах - заполнение нулями (если отсутствуют)
data['NA_Sales'].fillna(0, inplace=True)
data['EU_Sales'].fillna(0, inplace=True)
data['Other_Sales'].fillna(0, inplace=True)

print(f"  ✓ Все пропущенные значения обработаны")


[2/10] Предобработка данных...
  ✓ Удалено записей без JP_Sales: 0
  ✓ Записей после удаления критических пропусков: 11702
  ✓ Все пропущенные значения обработаны


4. СОЗДАНИЕ ПРИЗНАКОВ (FEATURE ENGINEERING)

In [16]:
print("\n[3/10] Создание дополнительных признаков...")

# ВАЖНО: Продажи в других регионах - ключевой признак для Японии
# Согласно задаче, мы используем историю продаж в других регионах
data['Total_Other_Sales'] = data['NA_Sales'] + data['EU_Sales'] + data['Other_Sales']

# Признаки на основе оценок
data['Has_Critic_Score'] = (data['Critic_Score'] > 0).astype(int)
data['Has_User_Score'] = (data['User_Score'] > 0).astype(int)
data['Score_Difference'] = data['Critic_Score'] / 10 - data['User_Score']
data['Total_Reviews'] = data['Critic_Count'] + data['User_Count']
data['Review_Intensity'] = data['Total_Reviews'] / (data['Critic_Score'] + 1)

# Популярность платформы в Японии (специфика японского рынка)
platform_jp_popularity = data.groupby('Platform')['JP_Sales'].sum().to_dict()
data['Platform_JP_Popularity'] = data['Platform'].map(platform_jp_popularity)

# Популярность жанра в Японии
genre_jp_popularity = data.groupby('Genre')['JP_Sales'].sum().to_dict()
data['Genre_JP_Popularity'] = data['Genre'].map(genre_jp_popularity)

# Средние продажи издателя в Японии
publisher_jp_avg = data.groupby('Publisher')['JP_Sales'].mean().to_dict()
data['Publisher_JP_Avg'] = data['Publisher'].map(publisher_jp_avg)

# Средние продажи разработчика в Японии
developer_jp_avg = data.groupby('Developer')['JP_Sales'].mean().to_dict()
data['Developer_JP_Avg'] = data['Developer'].map(developer_jp_avg)

# Возраст игры (может влиять на продажи в Японии)
current_year = data['Year_of_Release'].max()
data['Game_Age'] = current_year - data['Year_of_Release']

# Соотношение продаж NA к EU (паттерн для прогноза JP)
data['NA_EU_Ratio'] = data['NA_Sales'] / (data['EU_Sales'] + 0.01)

# Является ли игра японской (японские игры лучше продаются в Японии)
japanese_publishers = ['Nintendo', 'Sony Computer Entertainment', 'Namco Bandai Games',
                       'Konami Digital Entertainment', 'Capcom', 'Square Enix',
                       'Sega', 'Bandai', 'Tecmo Koei']
data['Is_Japanese_Publisher'] = data['Publisher'].isin(japanese_publishers).astype(int)

print(f"  ✓ Создано дополнительных признаков")


[3/10] Создание дополнительных признаков...
  ✓ Создано дополнительных признаков


5. КОДИРОВАНИЕ КАТЕГОРИАЛЬНЫХ ПРИЗНАКОВ

In [17]:
print("\n[4/10] Кодирование категориальных признаков...")

categorical_features = ['Platform', 'Genre', 'Publisher', 'Developer', 'Rating']

# Группировка редких категорий для оптимизации
# Publisher - топ-50 + Other
publisher_counts = data['Publisher'].value_counts()
top_publishers = publisher_counts.head(50).index
data['Publisher'] = data['Publisher'].apply(lambda x: x if x in top_publishers else 'Other')

# Developer - топ-100 + Other
developer_counts = data['Developer'].value_counts()
top_developers = developer_counts.head(100).index
data['Developer'] = data['Developer'].apply(lambda x: x if x in top_developers else 'Other')

# Label Encoding
label_encoders = {}
for col in categorical_features:
    le = LabelEncoder()
    data[col + '_encoded'] = le.fit_transform(data[col].astype(str))
    label_encoders[col] = le

print(f"  ✓ Категориальные признаки закодированы")


[4/10] Кодирование категориальных признаков...
  ✓ Категориальные признаки закодированы


6. ПОДГОТОВКА ДАННЫХ ДЛЯ ОБУЧЕНИЯ

In [18]:
print("\n[5/10] Подготовка данных для обучения...")

# Выбор признаков
numerical_features = [
    'Year_of_Release',
    'NA_Sales',              # КЛЮЧЕВОЙ признак
    'EU_Sales',              # КЛЮЧЕВОЙ признак
    'Other_Sales',           # КЛЮЧЕВОЙ признак
    'Total_Other_Sales',     # КЛЮЧЕВОЙ признак
    'Critic_Score',
    'Critic_Count',
    'User_Score',
    'User_Count',
    'Has_Critic_Score',
    'Has_User_Score',
    'Score_Difference',
    'Total_Reviews',
    'Review_Intensity',
    'Platform_JP_Popularity',
    'Genre_JP_Popularity',
    'Publisher_JP_Avg',
    'Developer_JP_Avg',
    'Game_Age',
    'NA_EU_Ratio',
    'Is_Japanese_Publisher'
]

encoded_features = [col + '_encoded' for col in categorical_features]
all_features = numerical_features + encoded_features

# Формирование матриц X и y
X = data[all_features].copy()
y = data['JP_Sales'].copy()

# Сохранение индексов для финального файла
data_ids = data.reset_index(drop=True).index + 1

print(f"  ✓ Размер датасета: {X.shape}")
print(f"  ✓ Количество признаков: {X.shape[1]}")
print(f"  ✓ Целевая переменная: JP_Sales")


[5/10] Подготовка данных для обучения...
  ✓ Размер датасета: (11702, 26)
  ✓ Количество признаков: 26
  ✓ Целевая переменная: JP_Sales


7. РАЗДЕЛЕНИЕ ДАННЫХ

In [19]:
print("\n[6/10] Разделение на обучающую и тестовую выборки...")

# Разделение 80/20
X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y, data_ids, test_size=0.2, random_state=42
)

# Масштабирование признаков
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"  ✓ Обучающая выборка: {X_train.shape[0]} записей")
print(f"  ✓ Тестовая выборка: {X_test.shape[0]} записей")


[6/10] Разделение на обучающую и тестовую выборки...
  ✓ Обучающая выборка: 9361 записей
  ✓ Тестовая выборка: 2341 записей


8. ОБУЧЕНИЕ МОДЕЛЕЙ

In [20]:
print("\n[7/10] Обучение моделей...")
print("  Метрика оптимизации: Mean Absolute Error (MAE)")
print()

models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Lasso Regression': Lasso(alpha=0.01),
    'ElasticNet': ElasticNet(alpha=0.01, l1_ratio=0.5),
    'Decision Tree': DecisionTreeRegressor(max_depth=10, random_state=42),
    'Random Forest': RandomForestRegressor(
        n_estimators=200,
        max_depth=15,
        min_samples_split=5,
        min_samples_leaf=2,
        n_jobs=-1,
        random_state=42
    ),
    'Extra Trees': ExtraTreesRegressor(
        n_estimators=200,
        max_depth=15,
        min_samples_split=5,
        n_jobs=-1,
        random_state=42
    ),
    'Gradient Boosting': GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=5,
        random_state=42
    ),
    'KNN': KNeighborsRegressor(n_neighbors=10)
}

results = {}
best_model_name = None
best_mae = float('inf')

for name, model in models.items():
    print(f"  Обучение {name}...")

    # Обучение
    model.fit(X_train_scaled, y_train)

    # Прогнозы
    y_pred_train = model.predict(X_train_scaled)
    y_pred_test = model.predict(X_test_scaled)

    # Метрики
    train_mae = mean_absolute_error(y_train, y_pred_train)
    test_mae = mean_absolute_error(y_test, y_pred_test)
    train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_train))
    test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
    train_r2 = r2_score(y_train, y_pred_train)
    test_r2 = r2_score(y_test, y_pred_test)

    results[name] = {
        'model': model,
        'train_mae': train_mae,
        'test_mae': test_mae,
        'train_rmse': train_rmse,
        'test_rmse': test_rmse,
        'train_r2': train_r2,
        'test_r2': test_r2
    }

    print(f"    Train MAE: {train_mae:.4f} | Test MAE: {test_mae:.4f}")
    print(f"    Train R²: {train_r2:.4f} | Test R²: {test_r2:.4f}")

    # Отслеживание лучшей модели по MAE
    if test_mae < best_mae:
        best_mae = test_mae
        best_model_name = name

print(f"\n  ✓ Лучшая модель: {best_model_name} (MAE: {best_mae:.4f})")


[7/10] Обучение моделей...
  Метрика оптимизации: Mean Absolute Error (MAE)

  Обучение Linear Regression...
    Train MAE: 0.0953 | Test MAE: 0.0999
    Train R²: 0.4269 | Test R²: 0.4787
  Обучение Ridge Regression...
    Train MAE: 0.0953 | Test MAE: 0.0999
    Train R²: 0.4269 | Test R²: 0.4787
  Обучение Lasso Regression...
    Train MAE: 0.0874 | Test MAE: 0.0921
    Train R²: 0.4149 | Test R²: 0.4677
  Обучение ElasticNet...
    Train MAE: 0.0909 | Test MAE: 0.0956
    Train R²: 0.4221 | Test R²: 0.4740
  Обучение Decision Tree...
    Train MAE: 0.0358 | Test MAE: 0.0679
    Train R²: 0.8858 | Test R²: 0.4479
  Обучение Random Forest...
    Train MAE: 0.0319 | Test MAE: 0.0565
    Train R²: 0.8652 | Test R²: 0.6344
  Обучение Extra Trees...
    Train MAE: 0.0262 | Test MAE: 0.0575
    Train R²: 0.9393 | Test R²: 0.6282
  Обучение Gradient Boosting...
    Train MAE: 0.0289 | Test MAE: 0.0586
    Train R²: 0.9406 | Test R²: 0.6397
  Обучение KNN...
    Train MAE: 0.0517 | Test MA

9. ОПТИМИЗАЦИЯ ЛУЧШЕЙ МОДЕЛИ

In [21]:
print(f"\n[8/10] Оптимизация модели {best_model_name}...")

if 'Random Forest' in best_model_name or 'Extra Trees' in best_model_name:
    param_grid = {
        'n_estimators': [200, 300],
        'max_depth': [15, 20],
        'min_samples_split': [3, 5],
        'min_samples_leaf': [1, 2]
    }
    if 'Random Forest' in best_model_name:
        grid_model = RandomForestRegressor(random_state=42, n_jobs=-1)
    else:
        grid_model = ExtraTreesRegressor(random_state=42, n_jobs=-1)

elif 'Gradient Boosting' in best_model_name:
    param_grid = {
        'n_estimators': [200, 300],
        'learning_rate': [0.05, 0.1],
        'max_depth': [4, 5, 6],
        'min_samples_split': [5, 10]
    }
    grid_model = GradientBoostingRegressor(random_state=42)
else:
    grid_model = results[best_model_name]['model']
    param_grid = None

if param_grid:
    print("  Запуск GridSearchCV (это может занять время)...")
    grid_search = GridSearchCV(
        grid_model,
        param_grid,
        cv=5,
        scoring='neg_mean_absolute_error',
        n_jobs=-1,
        verbose=1
    )

    grid_search.fit(X_train_scaled, y_train)

    final_model = grid_search.best_estimator_
    print(f"  ✓ Лучшие параметры: {grid_search.best_params_}")
else:
    final_model = grid_model

# Финальная оценка
y_pred_final_train = final_model.predict(X_train_scaled)
y_pred_final_test = final_model.predict(X_test_scaled)

final_train_mae = mean_absolute_error(y_train, y_pred_final_train)
final_test_mae = mean_absolute_error(y_test, y_pred_final_test)
final_test_r2 = r2_score(y_test, y_pred_final_test)

print(f"\n  ФИНАЛЬНЫЕ МЕТРИКИ:")
print(f"    Train MAE: {final_train_mae:.4f}")
print(f"    Test MAE: {final_test_mae:.4f}")
print(f"    Test R²: {final_test_r2:.4f}")


[8/10] Оптимизация модели Random Forest...
  Запуск GridSearchCV (это может занять время)...
Fitting 5 folds for each of 16 candidates, totalling 80 fits
  ✓ Лучшие параметры: {'max_depth': 20, 'min_samples_leaf': 2, 'min_samples_split': 3, 'n_estimators': 300}

  ФИНАЛЬНЫЕ МЕТРИКИ:
    Train MAE: 0.0276
    Test MAE: 0.0560
    Test R²: 0.6385


9.1. СТРАТЕГИЯ 1: ЛОГАРИФМИЧЕСКАЯ ТРАНСФОРМАЦИЯ

In [22]:
print("\n" + "="*80)
print("СТРАТЕГИЯ 1: ЛОГАРИФМИЧЕСКАЯ ТРАНСФОРМАЦИЯ")
print("="*80)

print("\nАнализ распределения целевой переменной:")
print(f"  Skewness (асимметрия): {y_train.skew():.2f}")
print(f"  Kurtosis (эксцесс): {y_train.kurtosis():.2f}")
print(f"  Среднее: {y_train.mean():.4f} млн")
print(f"  Медиана: {y_train.median():.4f} млн")
print(f"  Std Dev: {y_train.std():.4f} млн")

print(f"\n⚠️  ПРОБЛЕМА: Сильно правостороннее распределение (skewness = {y_train.skew():.2f})")
print("   Норма для нормального распределения: skewness ≈ 0")

# Применение логарифмической трансформации
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

print(f"\nПосле log1p трансформации:")
print(f"  Skewness: {y_train_log.skew():.2f}")
print(f"  Kurtosis: {y_train_log.kurtosis():.2f}")
print(f"  Среднее: {y_train_log.mean():.4f}")
print(f"  Std Dev: {y_train_log.std():.4f}")

skew_improvement = ((y_train.skew() - y_train_log.skew()) / y_train.skew()) * 100
print(f"\n✅ РЕЗУЛЬТАТ:")
print(f"   Skewness снижен на {skew_improvement:.1f}%")
print(f"   Распределение стало более нормальным и подходящим для ML-моделей")


СТРАТЕГИЯ 1: ЛОГАРИФМИЧЕСКАЯ ТРАНСФОРМАЦИЯ

Анализ распределения целевой переменной:
  Skewness (асимметрия): 12.32
  Kurtosis (эксцесс): 237.25
  Среднее: 0.0774 млн
  Медиана: 0.0000 млн
  Std Dev: 0.3155 млн

⚠️  ПРОБЛЕМА: Сильно правостороннее распределение (skewness = 12.32)
   Норма для нормального распределения: skewness ≈ 0

После log1p трансформации:
  Skewness: 5.62
  Kurtosis: 43.25
  Среднее: 0.0565
  Std Dev: 0.1590

✅ РЕЗУЛЬТАТ:
   Skewness снижен на 54.4%
   Распределение стало более нормальным и подходящим для ML-моделей


9.2. СТРАТЕГИЯ 2: УСТРАНЕНИЕ ПЕРЕОБУЧЕНИЯ

In [23]:
print("\n" + "="*80)
print("СТРАТЕГИЯ 2: УСТРАНЕНИЕ ПЕРЕОБУЧЕНИЯ")
print("="*80)

print("\nАнализ переобучения базовых моделей:")
print(f"{'Модель':<30} {'Train R²':<12} {'Test R²':<12} {'ΔR²':<10} {'Статус'}")
print("-"*80)

for name in results.keys():
    train_r2 = results[name]['train_r2']
    test_r2 = results[name]['test_r2']
    delta_r2 = train_r2 - test_r2

    if delta_r2 > 0.3:
        status = "❌ Сильное"
    elif delta_r2 > 0.2:
        status = "⚠️  Умеренное"
    else:
        status = "✅ Нормально"

    print(f"{name:<30} {train_r2:<12.4f} {test_r2:<12.4f} {delta_r2:<10.4f} {status}")

baseline_overfit = results[best_model_name]['train_r2'] - results[best_model_name]['test_r2']
print(f"\n⚠️  ПРОБЛЕМА: Лучшая модель имеет переобучение ΔR² = {baseline_overfit:.4f}")

print("\nОптимизированные гиперпараметры:")
print("-"*80)

# Определение оптимизированных параметров на основе типа модели
if 'Random Forest' in best_model_name:
    print("\n📋 Random Forest - Оптимизированные параметры:")
    print("   n_estimators:      200 → 300  (больше деревьев для стабильности)")
    print("   max_depth:          15 → 12   (меньше глубина → меньше переобучение)")
    print("   min_samples_split:   5 → 10   (больше образцов для split)")
    print("   min_samples_leaf:    2 → 5    (больше образцов в листе)")
    print("   max_features:     None → 'sqrt' (ограничение случайных признаков)")
    print("   bootstrap:       True → True")
    print("   oob_score:      False → True  (out-of-bag оценка)")

    optimized_model = RandomForestRegressor(
        n_estimators=300,
        max_depth=12,
        min_samples_split=10,
        min_samples_leaf=5,
        max_features='sqrt',
        bootstrap=True,
        oob_score=True,
        n_jobs=-1,
        random_state=42
    )

elif 'Extra Trees' in best_model_name:
    print("\n📋 Extra Trees - Оптимизированные параметры:")
    print("   n_estimators:      200 → 300")
    print("   max_depth:          15 → 12")
    print("   min_samples_split:   5 → 10")
    print("   min_samples_leaf:    2 → 5")
    print("   max_features:     None → 'sqrt'")
    print("   bootstrap:      False → True  (добавлена подвыборка)")
    print("   oob_score:      False → True")

    optimized_model = ExtraTreesRegressor(
        n_estimators=300,
        max_depth=12,
        min_samples_split=10,
        min_samples_leaf=5,
        max_features='sqrt',
        bootstrap=True,
        oob_score=True,
        n_jobs=-1,
        random_state=42
    )

elif 'Gradient Boosting' in best_model_name:
    print("\n📋 Gradient Boosting - Оптимизированные параметры:")
    print("   n_estimators:       200 → 300")
    print("   learning_rate:      0.1 → 0.05  (медленнее обучение → лучше генерализация)")
    print("   max_depth:            5 → 4     (меньше сложность)")
    print("   min_samples_split:    5 → 10")
    print("   min_samples_leaf:     2 → 5")
    print("   subsample:          1.0 → 0.8   (стохастический GB)")
    print("   validation_fraction:  - → 0.1   (для early stopping)")
    print("   n_iter_no_change:     - → 20    (early stopping)")

    optimized_model = GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        min_samples_split=10,
        min_samples_leaf=5,
        subsample=0.8,
        validation_fraction=0.1,
        n_iter_no_change=20,
        random_state=42
    )

else:
    # Для других моделей (Ridge, Lasso, etc.) используем базовую версию с логарифмом
    print(f"\n📋 {best_model_name} - использование базовой конфигурации с логарифмом")
    # Берем модель из словаря results
    optimized_model = results[best_model_name]['model']

print("\n✅ ЦЕЛЬ: Снизить переобучение при сохранении точности")


СТРАТЕГИЯ 2: УСТРАНЕНИЕ ПЕРЕОБУЧЕНИЯ

Анализ переобучения базовых моделей:
Модель                         Train R²     Test R²      ΔR²        Статус
--------------------------------------------------------------------------------
Linear Regression              0.4269       0.4787       -0.0519    ✅ Нормально
Ridge Regression               0.4269       0.4787       -0.0519    ✅ Нормально
Lasso Regression               0.4149       0.4677       -0.0529    ✅ Нормально
ElasticNet                     0.4221       0.4740       -0.0519    ✅ Нормально
Decision Tree                  0.8858       0.4479       0.4379     ❌ Сильное
Random Forest                  0.8652       0.6344       0.2308     ⚠️  Умеренное
Extra Trees                    0.9393       0.6282       0.3111     ❌ Сильное
Gradient Boosting              0.9406       0.6397       0.3009     ❌ Сильное
KNN                            0.6373       0.5799       0.0574     ✅ Нормально

⚠️  ПРОБЛЕМА: Лучшая модель имеет переобучение ΔR² 

In [24]:
9.3. СТРАТЕГИЯ 3: ОБРАБОТКА ДИСБАЛАНСА ВЕСАМИ

SyntaxError: invalid syntax (ipython-input-3026378187.py, line 1)

In [ ]:
print("\n" + "="*80)
print("СТРАТЕГИЯ 3: ОБРАБОТКА ДИСБАЛАНСА ВЕСАМИ")
print("="*80)

# Анализ дисбаланса
zero_sales = (y_train == 0).sum()
low_sales = ((y_train > 0) & (y_train <= 0.1)).sum()
medium_sales = ((y_train > 0.1) & (y_train <= 0.5)).sum()
high_sales = (y_train > 0.5).sum()
total_train = len(y_train)

print("\nАнализ распределения JP_Sales в обучающей выборке:")
print("-"*80)
print(f"{'Диапазон':<25} {'Количество':<12} {'Процент':<12} {'Визуализация'}")
print("-"*80)
print(f"{'Нулевые (0 млн)':<25} {zero_sales:<12} {zero_sales/total_train*100:>6.1f}%     {'█' * min(int(zero_sales/total_train*50), 50)}")
print(f"{'Низкие (0-0.1 млн)':<25} {low_sales:<12} {low_sales/total_train*100:>6.1f}%     {'█' * min(int(low_sales/total_train*50), 50)}")
print(f"{'Средние (0.1-0.5 млн)':<25} {medium_sales:<12} {medium_sales/total_train*100:>6.1f}%     {'█' * min(int(medium_sales/total_train*50), 50)}")
print(f"{'Высокие (>0.5 млн)':<25} {high_sales:<12} {high_sales/total_train*100:>6.1f}%     {'█' * min(int(high_sales/total_train*50), 50)}")
print("-"*80)

print(f"\n⚠️  ПРОБЛЕМА: Крайний дисбаланс!")
print(f"   • {zero_sales/total_train*100:.1f}% игр имеют нулевые продажи в Японии")
print(f"   • Только {high_sales/total_train*100:.1f}% игр имеют высокие продажи (>0.5 млн)")
print(f"   • Модель будет склонна игнорировать редкие высокие продажи")

# Функция для вычисления весов
def compute_sample_weights(y_values):
    """
    Вычисляет веса для образцов на основе диапазонов продаж.
    Редкие события (высокие продажи) получают больший вес.
    """
    weights = np.ones(len(y_values))
    y_array = y_values.values if hasattr(y_values, 'values') else y_values

    # Определение диапазонов
    zero_mask = y_array == 0
    low_mask = (y_array > 0) & (y_array <= 0.1)
    medium_mask = (y_array > 0.1) & (y_array <= 0.5)
    high_mask = y_array > 0.5

    # Подсчет частот
    n_zero = zero_mask.sum()
    n_low = low_mask.sum()
    n_medium = medium_mask.sum()
    n_high = high_mask.sum()
    n_total = len(y_array)

    # Вычисление весов (обратно пропорционально частоте)
    if n_zero > 0:
        weights[zero_mask] = 1.0  # Базовый вес
    if n_low > 0:
        weights[low_mask] = n_total / (4 * n_low)
    if n_medium > 0:
        weights[medium_mask] = n_total / (4 * n_medium)
    if n_high > 0:
        weights[high_mask] = n_total / (4 * n_high)

    # Нормализация весов
    weights = weights / weights.mean()

    return weights

# Вычисление весов
print("\nВычисление весов...")
sample_weights = compute_sample_weights(y_train)

print("\n📊 Статистика весов:")
print(f"   Минимальный вес: {sample_weights.min():.2f}")
print(f"   Средний вес:     {sample_weights.mean():.2f}")
print(f"   Максимальный вес: {sample_weights.max():.2f}")
print(f"   Медианный вес:   {np.median(sample_weights):.2f}")

# Средние веса по группам
print(f"\n📈 Средние веса по группам продаж:")
print("-"*80)
if zero_sales > 0:
    zero_weight = sample_weights[y_train == 0].mean()
    print(f"   Нулевые продажи:  {zero_weight:6.2f}  (базовый вес)")
if low_sales > 0:
    low_weight = sample_weights[(y_train > 0) & (y_train <= 0.1)].mean()
    print(f"   Низкие продажи:   {low_weight:6.2f}  (×{low_weight/zero_weight:.1f} раз больше)")
if medium_sales > 0:
    medium_weight = sample_weights[(y_train > 0.1) & (y_train <= 0.5)].mean()
    print(f"   Средние продажи:  {medium_weight:6.2f}  (×{medium_weight/zero_weight:.1f} раз больше)")
if high_sales > 0:
    high_weight = sample_weights[y_train > 0.5].mean()
    print(f"   Высокие продажи:  {high_weight:6.2f}  (×{high_weight/zero_weight:.1f} раз больше)")

print("\n✅ РЕШЕНИЕ: Веса вычислены!")

9.4. ОБУЧЕНИЕ ОПТИМИЗИРОВАННОЙ МОДЕЛИ

In [ ]:
print("\n" + "="*80)
print("ОБУЧЕНИЕ ОПТИМИЗИРОВАННОЙ МОДЕЛИ")
print("="*80)

print("\n🎯 Применяемые стратегии:")
print("   ✅ Логарифмическая трансформация целевой переменной")
print("   ✅ Оптимизированные гиперпараметры (против переобучения)")
print("   ✅ Взвешенное обучение (компенсация дисбаланса)")

print("\n⏳ Запуск обучения оптимизированной модели...")

# Обучение с весами
try:
    optimized_model.fit(X_train_scaled, y_train_log, sample_weight=sample_weights)
    print("   ✓ Модель обучена с sample_weight")
    used_weights = True
except:
    print("   ⚠️  Обучение без весов")
    optimized_model.fit(X_train_scaled, y_train_log)
    used_weights = False

# Прогнозы
y_pred_train_log = optimized_model.predict(X_train_scaled)
y_pred_test_log = optimized_model.predict(X_test_scaled)

# Обратная трансформация
y_pred_train_optimized = np.expm1(y_pred_train_log)
y_pred_test_optimized = np.expm1(y_pred_test_log)
y_pred_train_optimized = np.clip(y_pred_train_optimized, 0, None)
y_pred_test_optimized = np.clip(y_pred_test_optimized, 0, None)

# Метрики
optimized_train_mae = mean_absolute_error(y_train, y_pred_train_optimized)
optimized_test_mae = mean_absolute_error(y_test, y_pred_test_optimized)
optimized_train_r2 = r2_score(y_train, y_pred_train_optimized)
optimized_test_r2 = r2_score(y_test, y_pred_test_optimized)

# Сравнение
print("\n" + "="*80)
print("РЕЗУЛЬТАТЫ ОПТИМИЗАЦИИ")
print("="*80)

baseline_train_mae = results[best_model_name]['train_mae']
baseline_test_mae = results[best_model_name]['test_mae']
baseline_train_r2 = results[best_model_name]['train_r2']
baseline_test_r2 = results[best_model_name]['test_r2']

mae_improvement = ((baseline_test_mae - optimized_test_mae) / baseline_test_mae) * 100
overfit_base = baseline_train_r2 - baseline_test_r2
overfit_opt = optimized_train_r2 - optimized_test_r2

print(f"\n{'Метрика':<25} {'Базовая':<15} {'Оптимизированная':<18} {'Изменение':<15}")
print("="*80)
print(f"{'Test MAE (млн)':<25} {baseline_test_mae:<15.4f} {optimized_test_mae:<18.4f} {optimized_test_mae - baseline_test_mae:<+15.4f}")
print(f"{'Test R²':<25} {baseline_test_r2:<15.4f} {optimized_test_r2:<18.4f} {optimized_test_r2 - baseline_test_r2:<+15.4f}")
print(f"{'Переобучение (ΔR²)':<25} {overfit_base:<15.4f} {overfit_opt:<18.4f} {overfit_opt - overfit_base:<+15.4f}")

print(f"\n✨ УЛУЧШЕНИЕ: {mae_improvement:+.2f}%")
print(f"   Снижение ошибки на {(baseline_test_mae - optimized_test_mae)*1_000_000:,.0f} копий")

# Сохранение финальной модели
final_model = optimized_model

print(f"\n🚀 ФИНАЛЬНАЯ МОДЕЛЬ ГОТОВА!")
print("="*80)

10. СОЗДАНИЕ ФИНАЛЬНОГО ПРОГНОЗА

In [ ]:
print(f"\n[9/10] Создание файла с прогнозами...")

# Прогноз для всех данных
X_all_scaled = scaler.transform(X)
y_pred_all = final_model.predict(X_all_scaled)

# Создание результирующего DataFrame
result_df = pd.DataFrame({
    'Id': data_ids,
    'JP_Sales': y_pred_all
})

# Округление до 2 знаков (как в примере)
result_df['JP_Sales'] = result_df['JP_Sales'].round(2)

# Убедимся, что нет отрицательных значений
result_df['JP_Sales'] = result_df['JP_Sales'].clip(lower=0)

# Сохранение в CSV
output_path = '/content/sample_data/predictions_JP_Sales.csv'
result_df.to_csv(output_path, index=False)

print(f"  ✓ Файл сохранен: {output_path}")
print(f"  ✓ Количество прогнозов: {len(result_df)}")
print(f"\n  Первые строки результата:")
print(result_df.head(10))

11. ВИЗУАЛИЗАЦИЯ И ОТЧЕТ

In [ ]:
print(f"\n[10/10] Создание визуализаций и отчета...")

# График 1: Сравнение моделей
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

model_names = list(results.keys())
test_mae_values = [results[name]['test_mae'] for name in model_names]
test_r2_values = [results[name]['test_r2'] for name in model_names]

axes[0].barh(model_names, test_mae_values, color='skyblue')
axes[0].set_xlabel('MAE (меньше = лучше)', fontsize=12)
axes[0].set_title('Сравнение моделей по MAE', fontsize=12, fontweight='bold')
axes[0].grid(axis='x', alpha=0.3)
axes[0].invert_yaxis()

axes[1].barh(model_names, test_r2_values, color='lightcoral')
axes[1].set_xlabel('R² Score (больше = лучше)', fontsize=12)
axes[1].set_title('Сравнение моделей по R²', fontsize=12, fontweight='bold')
axes[1].grid(axis='x', alpha=0.3)
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig('/content/sample_data/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# График 2: Фактические vs Прогнозируемые
plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred_final_test, alpha=0.5, s=30, edgecolors='black', linewidth=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Идеальный прогноз')
plt.xlabel('Фактические продажи в Японии (млн)', fontsize=12)
plt.ylabel('Прогнозируемые продажи (млн)', fontsize=12)
plt.title('Фактические vs Прогнозируемые продажи в Японии', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/content/sample_data/actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()

# График 3: Важность признаков
if hasattr(final_model, 'feature_importances_'):
    feature_importance = pd.DataFrame({
        'feature': all_features,
        'importance': final_model.feature_importances_
    }).sort_values('importance', ascending=False).head(20)

    plt.figure(figsize=(10, 8))
    plt.barh(feature_importance['feature'], feature_importance['importance'], color='mediumseagreen')
    plt.xlabel('Важность', fontsize=12)
    plt.title('Топ-20 важнейших признаков для прогноза JP_Sales', fontsize=14, fontweight='bold')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.savefig('/content/sample_data/feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()

    print("\n  ТОП-10 ВАЖНЕЙШИХ ПРИЗНАКОВ:")
    for idx, row in feature_importance.head(10).iterrows():
        print(f"    {row['feature']}: {row['importance']:.4f}")

# График 4: Распределение ошибок
errors = y_test - y_pred_final_test
plt.figure(figsize=(10, 6))
plt.hist(errors, bins=50, edgecolor='black', alpha=0.7, color='steelblue')
plt.xlabel('Ошибка прогноза (млн)', fontsize=12)
plt.ylabel('Частота', fontsize=12)
plt.title('Распределение ошибок прогноза', fontsize=14, fontweight='bold')
plt.axvline(x=0, color='red', linestyle='--', linewidth=2, label='Нулевая ошибка')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/content/sample_data/error_distribution.png', dpi=150, bbox_inches='tight')
plt.show()



---


# **ИТОГОВЫЙ ОТЧЕТ**


In [ ]:
print("\n" + "="*80)
print("ИТОГОВЫЙ ОТЧЕТ: ПРОГНОЗИРОВАНИЕ ПРОДАЖ В ЯПОНИИ")
print("="*80)

print("\n1. СРАВНИТЕЛЬНАЯ ТАБЛИЦА МОДЕЛЕЙ:")
print("-"*80)
results_table = pd.DataFrame({
    'Модель': model_names,
    'Train MAE': [results[name]['train_mae'] for name in model_names],
    'Test MAE': [results[name]['test_mae'] for name in model_names],
    'Train R²': [results[name]['train_r2'] for name in model_names],
    'Test R²': [results[name]['test_r2'] for name in model_names]
}).round(4)
print(results_table.to_string(index=False))

print("\n2. ФИНАЛЬНАЯ МОДЕЛЬ:")
print("-"*80)
print(f"  Название: {best_model_name}")
print(f"  Test MAE: {final_test_mae:.4f} млн копий")
print(f"  Test R²: {final_test_r2:.4f}")
print(f"\n  Интерпретация MAE:")
print(f"  В среднем модель ошибается на ±{final_test_mae*1000000:.0f} копий")
print(f"  при прогнозировании продаж в Японии")

print("\n3. КЛЮЧЕВЫЕ ФАКТОРЫ УСПЕХА В ЯПОНИИ:")
print("-"*80)
print("  • Продажи в других регионах (NA, EU) - сильный предиктор")
print("  • Популярность платформы на японском рынке")
print("  • Популярность жанра (RPG, Platform, Action)")
print("  • Принадлежность к японскому издателю")
print("  • Оценки критиков и пользователей")

print("\n4. ОСОБЕННОСТИ ЯПОНСКОГО РЫНКА:")
print("-"*80)
print("  • Консольные игры доминируют (Nintendo, PlayStation)")
print("  • Сильное предпочтение локальных франшиз")
print("  • RPG и платформеры - наиболее популярные жанры")
print("  • Высокая лояльность к японским издателям")
print("  • Мобильные и казуальные игры растут в популярности")

print("\n5. СОЗДАННЫЕ ФАЙЛЫ:")
print("-"*80)
print(f"  ✓ predictions_JP_Sales.csv - прогнозы для всех игр")
print(f"  ✓ model_comparison.png - сравнение моделей")
print(f"  ✓ actual_vs_predicted.png - качество прогнозов")
print(f"  ✓ feature_importance.png - важность признаков")
print(f"  ✓ error_distribution.png - распределение ошибок")

print("\n" + "="*80)
print("АНАЛИЗ ЗАВЕРШЕН УСПЕШНО")
print("="*80)

# Скачивание результирующего файла
print("\nСкачивание файла с прогнозами...")
from google.colab import files
files.download('/content/sample_data/predictions_JP_Sales.csv')